## 5.5 Pytorch 模拟线性回归 - 多特征输入

#### 1. 从单特征到多特征：概念升级 
* 之前我们做的是：`y = w * x + b`
* 现在升级为：`y = w1*x1 + w2* x2 + ... + wn * xn + b`
* 在本案例中：
    * X shape = (N, 3)
    * W shape = (3, 1)
    * b shape = (1)

#### 2.第一步：重新设计数据（规模扩大）

我们构造：
* 1000 个样本
* 每个样本 3 个特征

##### 2.1 构造特征矩阵X

In [1]:
import torch

torch.manual_seed(42)  # 固定随机种子，方便复现

N = 1000 # 样本数量
D = 3 # 特征数量
X = torch.randn(N, D) # 生成随机数据

##### 2.2 设定真实参数（我们希望模型学出来）

In [4]:
true_w = torch.tensor([[2.0, -3, 1.5]]) # 真实权重,shape=(1, D)
true_b = torch.tensor([0.5]) # 真实偏置,shape=(1,)

##### 2.3 加入噪声，生成y

In [5]:
noise = torch.randn(N, 1) * 0.5 # 添加一些噪声
y = X @ true_w.T + true_b + noise # 生成目标变量,shape=(N, 1)

##### 2.4 检查形状

In [ ]:
print(X.shape) # 输出: torch.Size([1000, 3])
print(y.shape) # 输出: torch.Size([1000, 1])

torch.Size([1000, 3])
torch.Size([1000, 1])


#### 3. 构造Mini-batch + DataLoader

In [7]:
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(X, y) # 创建数据集
batch_size = 32
dataloader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2
)

#### 4. 定义模型参数 - 多特征版本

In [8]:
import torch.nn as nn
model = nn.Linear(in_features=D, out_features=1) # 输入维度D，输出维度1

criterion = nn.MSELoss() # 均方误差损失函数
optimizer = torch.optim.SGD(model.parameters(), lr=0.05) # 随机

##### 4.1 ⚠️ 非常重要：

nn.Linear(3,1) 内部参数 shape：
* weight: (1,3)
* bias:   (1,)

注意：
* weight权重中的特征权重应该放在和X特征相同的维度，个数匹配
* 比如三个特征的X（N，3）， 对应3个权重的weight（1，3）


##### 4.2  为什么这么设计？
如果：`nn.Linear(3, 2)` ：
* 输入的特征数量为3个
* 输出的特征数量为2个

weight shape 会是：`(2,3)` :
* 第 1 行 → 第 1 个神经元的权重，
    * 与X的特征数量保持一致为3
* 第 2 行 → 第 2 个神经元的权重
    * 与X的特征数量保持一致为3

##### 4.3 举个具体例子
假设：<br>
`nn.Linear(3,2)`, `X.shape = (100,3)` <br>
此时：<br>
`w.shape(2,3)` <br>
因为：
* 输入的特征数量为3，所以w的特征维度的大小应该和X的特征维度大小相同，都为3
* 输出特征数量为2，所以w的个数维度的大小应该和输出特征数量保持一致

Linear 内部做：
```
X @ weight.T
(100,3) @ (3,2) = (100,2)
```
输出 `shape = (batch_size, out_features)`完全符合神经网络定义。


#### 5. 训练代码

In [10]:
epochs = 100
for epoch in range(1, epochs+1):
    for batch_X, batch_y in dataloader:
        # 前向传播
        y_pred = model(batch_X)
        # 计算损失
        loss = criterion(y_pred, batch_y)
        # 反向传播和优化
        loss.backward()
        # 更新参数
        optimizer.step()
        optimizer.zero_grad() # 清除梯度
        
    if epoch % 10 == 0:
        print(f'Epoch {epoch}/{epochs}, weight: {model.weight.data}, bias: {model.bias.data}, loss: {loss:.4f}')

Epoch 10/100, weight: tensor([[ 2.0006, -3.0143,  1.4881]]), bias: tensor([0.4908]), loss: 0.0960
Epoch 20/100, weight: tensor([[ 1.9986, -3.0061,  1.5177]]), bias: tensor([0.4655]), loss: 0.0739
Epoch 30/100, weight: tensor([[ 1.9376, -3.0142,  1.4728]]), bias: tensor([0.4857]), loss: 0.3007
Epoch 40/100, weight: tensor([[ 2.0201, -3.0289,  1.5046]]), bias: tensor([0.5023]), loss: 0.1584
Epoch 50/100, weight: tensor([[ 1.9949, -3.0439,  1.5100]]), bias: tensor([0.4969]), loss: 0.3961
Epoch 60/100, weight: tensor([[ 1.9919, -2.9978,  1.5148]]), bias: tensor([0.5115]), loss: 0.2863
Epoch 70/100, weight: tensor([[ 1.9975, -2.9960,  1.4786]]), bias: tensor([0.5393]), loss: 0.1317
Epoch 80/100, weight: tensor([[ 1.9802, -3.0137,  1.4887]]), bias: tensor([0.4876]), loss: 0.3357
Epoch 90/100, weight: tensor([[ 2.0078, -2.9749,  1.4719]]), bias: tensor([0.4846]), loss: 0.2810
Epoch 100/100, weight: tensor([[ 2.0087, -2.9988,  1.4434]]), bias: tensor([0.5040]), loss: 0.3956


#### 6. 训练完成后检查学到的参数

In [13]:
print("\nLearned parameters:")
print("W =", model.weight.data)
print("b =", model.bias.data)

print("\nTrue parameters:")
print("W =", true_w)
print("b =", true_b)


Learned parameters:
W = tensor([[ 2.0087, -2.9988,  1.4434]])
b = tensor([0.5040])

True parameters:
W = tensor([[ 2.0000, -3.0000,  1.5000]])
b = tensor([0.5000])
